# Submission 5 — Project Deployment (5%)

**Course:** RBB2013 Digital Twin — May 2026
**Group project — SmartClean Twin:** Digital Twin of a mobile cleaning robot (topic 2)

**Team Members:**

| No | Name | Student ID |
|---|---|---|
| 1 | Chan Li Kai | 22010900 |
| 2 | William Wong Xiao Kang | 22010943 |
| 3 | Irvin Chang Hou Ceng | 22012342 |
| 4 | Liang Yan Ee | 22011522 |
| 5 | Nurin Emelin Binti Marhisyam | 24006706 |
**Repository:** https://github.com/KAI-UTP/smartclean-twin

**Presentation & demo video:** [https://youtu.be/zEq7L-ivMLA](https://youtu.be/zEq7L-ivMLA)

> The video walks through the whole project: problem and purpose, architecture, live Grafana dashboard, NVIDIA Omniverse 3D twin, the five AI models, what-if simulation, live fault injection, command and control, tests, CI, scaling and persistence.


## 0. Presentation Video

**Full project presentation & demo video:**


## 1. Microservice partition — function of each service

| Service | Container | Port | Function |
|---|---|---|---|
| robot-simulator | smartclean-simulator | 8004 | physics @1s, telemetry publisher, command executor, fault-injection API |
| mosquitto | smartclean-mosquitto | 1883 | MQTT broker — all inter-service messaging |
| telemetry-ingestion | (scalable) | 8001 | schema validation, InfluxDB writer, republish validated |
| state-engine | smartclean-state-engine | 8002 | 11-dimension twin state, alarm rules |
| ai-service | smartclean-ai-service | 8003 | 5 ML models, forecasts, recommendations, /whatif |
| command-api | smartclean-command-api | 8000 | REST → MQTT commands, acknowledgement tracking |
| influxdb | smartclean-influxdb | 8086 | time-series persistence |
| grafana | smartclean-grafana | 3001 | dashboard (provisioned as code) |

## 2. Interface contract

`docs/api-contract.md` specifies, **for every pair of communicating
services**: route/topic, port, protocol (MQTT/HTTP), data format (JSON
schema), and when communication starts and ends (16 sections, including the
Omniverse→InfluxDB interface).

## 3. Containerization

Each microservice has its own Dockerfile; one-command deployment:
`docker compose up -d`. AI models are trained during image build (reproducible).


## 3. Interface contract — sample pairs (full contract: `docs/api-contract.md`)

`docs/api-contract.md` documents **all 16 communicating pairs**. For each pair it
states the route/topic, port, protocol, data format, and **when the communication
is initiated and concluded**. Three representative pairs are reproduced here.

**Pair 1 — robot-simulator → mosquitto**

| Field | Value |
|---|---|
| Route / topic | `smartclean/SCR01/telemetry/raw` |
| Port / protocol | 1883 / MQTT, QoS 1 |
| Data format | JSON, `TelemetryMessage` schema (16 sensor fields) |
| Initiated | On simulator startup, after the MQTT connect succeeds |
| Concluded | On SIGTERM / SIGINT received by the simulator |

**Pair 4 — telemetry-ingestion → influxdb**

| Field | Value |
|---|---|
| Route | `POST /api/v2/write?org=smartclean&bucket=smartclean_twin` |
| Port / protocol | 8086 / HTTP with Influx line protocol |
| Data format | Line protocol, measurement `robot_telemetry`, tag `robot_id` |
| Initiated | Per valid telemetry message (synchronous write) |
| Concluded | After the write acknowledgement, HTTP 204 |

**Pair 12 — command-api → mosquitto → robot-simulator**

| Field | Value |
|---|---|
| Route / topic | `smartclean/SCR01/command/motion`, reply on `smartclean/SCR01/ack` |
| Port / protocol | 1883 / MQTT, QoS 1 |
| Data format | JSON, `CommandMessage` out and `AckMessage` back |
| Initiated | When the Command API receives an HTTP POST from the operator |
| Concluded | When the ACK arrives from the simulator, or on timeout |

The remaining 13 pairs are documented in the same format in
`docs/api-contract.md`, together with a port summary table.


## 4. Live evidence — deployment status

In [1]:
import subprocess
r = subprocess.run(["docker", "compose", "ps", "--format", "{{.Name}}  {{.Status}}"],
                   capture_output=True, text=True, cwd=".")
print(r.stdout)


smartclean-ai-service  Up 4 minutes
smartclean-command-api  Up 4 minutes
smartclean-grafana  Up 4 minutes
smartclean-influxdb  Up About a minute (healthy)
smartclean-mosquitto  Up 55 seconds (healthy)
smartclean-simulator  Up 4 minutes
smartclean-state-engine  Up 4 minutes
smartclean-twin-telemetry-ingestion-1  Up 4 minutes



## 5. Live evidence — scaling a microservice

telemetry-ingestion is safely horizontally scalable (MQTT fan-out).
Scale to 2 instances, show both running, scale back.


In [2]:
import subprocess, time
subprocess.run(["docker", "compose", "up", "--scale", "telemetry-ingestion=2", "-d"],
               capture_output=True, text=True, cwd=".")
time.sleep(15)
r = subprocess.run(["docker", "compose", "ps", "--format", "{{.Name}}  {{.Status}}"],
                   capture_output=True, text=True, cwd=".")
print("With 2 ingestion instances:")
for line in r.stdout.splitlines():
    if "telemetry" in line:
        print(" ", line)
subprocess.run(["docker", "compose", "up", "--scale", "telemetry-ingestion=1", "-d"],
               capture_output=True, text=True, cwd=".")
print()
print("Scaled back to 1.")


With 2 ingestion instances:
  smartclean-twin-telemetry-ingestion-1  Up 4 minutes
  smartclean-twin-telemetry-ingestion-2  Up 15 seconds



Scaled back to 1.


## 6. Live evidence — persistence in data and state storage

The system test restarts the InfluxDB container and proves the data written
before the restart is still there afterwards.


In [3]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "tests/system/test_persistence.py", "-v"],
                   capture_output=True, text=True, cwd=".")
print(r.stdout[-1200:])
print("Exit code:", r.returncode, "(0 = persistence proven)")


============================= test session starts =============================
platform win32 -- Python 3.13.3, pytest-8.3.5, pluggy-1.6.0 -- C:\Users\TUF FA707RC-HX024W\AppData\Local\Programs\Python\Python313\python.exe
cachedir: .pytest_cache
hypothesis profile 'default'
rootdir: D:\UTP\UG Y2S3\03 Digital Twin\smartclean-twin
configfile: pyproject.toml
plugins: anyio-4.14.1, hypothesis-6.155.7, cov-7.1.0
collecting ... collected 3 items

tests/system/test_persistence.py::test_data_persists_after_influxdb_restart PASSED [ 33%]
tests/system/test_persistence.py::test_state_data_persists_after_influxdb_restart PASSED [ 66%]
tests/system/test_persistence.py::test_prediction_data_persists_after_influxdb_restart PASSED [100%]

============================= 3 passed in 12.62s ==============================

Exit code: 0 (0 = persistence proven)


## 7. Live evidence — full digital-twin flow test

End-to-end: telemetry published → validated → stored → twin state derived.


In [4]:
import os, subprocess, sys
env = dict(os.environ, INTEGRATION_TEST="1")
r = subprocess.run([sys.executable, "-m", "pytest", "tests/system/test_full_flow.py", "-v"],
                   capture_output=True, text=True, cwd=".", env=env)
print(r.stdout[-1500:])
print("Exit code:", r.returncode)


ts =============================
platform win32 -- Python 3.13.3, pytest-8.3.5, pluggy-1.6.0 -- C:\Users\TUF FA707RC-HX024W\AppData\Local\Programs\Python\Python313\python.exe
cachedir: .pytest_cache
hypothesis profile 'default'
rootdir: D:\UTP\UG Y2S3\03 Digital Twin\smartclean-twin
configfile: pyproject.toml
plugins: anyio-4.14.1, hypothesis-6.155.7, cov-7.1.0
collecting ... collected 11 items

tests/system/test_full_flow.py::TestServiceHealth::test_command_api_health PASSED [  9%]
tests/system/test_full_flow.py::TestServiceHealth::test_ingestion_health PASSED [ 18%]
tests/system/test_full_flow.py::TestServiceHealth::test_state_engine_health PASSED [ 27%]
tests/system/test_full_flow.py::TestServiceHealth::test_ai_service_health PASSED [ 36%]
tests/system/test_full_flow.py::TestServiceHealth::test_simulator_health PASSED [ 45%]
tests/system/test_full_flow.py::TestCommandFlow::test_pause_command_returns_ack PASSED [ 54%]
tests/system/test_full_flow.py::TestCommandFlow::test_resume_after